# LMTrade on Google Colab — demo runs & prod deploy

High-cadence hybrid trading bot: synthetic **options** (Black-Scholes) on live underlyings, an **evolutionary strategy learner**, **hourly Perplexity news**, a **daily Claude strategy review**, and a live **alpha benchmark vs buy-and-hold SPY** (the retail baseline).

Honesty first:
- **Paper mode only here.** Trade Republic has no official API; true HFT through it is impossible (its unofficial mobile API has seconds-to-minutes latency). This bot is *high-cadence intraday* (seconds-scale cycles), not HFT.
- **Outperformance is measured, never guaranteed.** The alpha panel tells you honestly whether the bot beats the retail baseline.
- Colab is for **demo runs**; the last section deploys a production instance to **Vast.ai**.

Run the cells top to bottom.

## 1. Install LMTrade

In [ ]:
!git clone https://github.com/269652/LMTrade.git
%cd LMTrade
!pip install -e . -q
!pip install -q yfinance   # live market data (synthetic fallback if closed)
print('\nLMTrade installed')

## 2. (Optional) Serve a local SLM with Ollama on the Colab GPU

Auto-sizes the model to your GPU: `qwen2.5:1.5b` on a T4, `qwen2.5:7b` on an A100. Skip this cell to run with the financial models + learned strategies only. Enable `Runtime -> Change runtime type -> GPU` first.

In [ ]:
import os, subprocess, time, shutil

# Pick the SLM by GPU size (T4 ~16GB -> 1.5b, A100 -> 7b)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
SLM_MODEL = 'qwen2.5:7b' if 'A100' in gpu else 'qwen2.5:1.5b'
print('GPU:', gpu or 'none', '-> SLM:', SLM_MODEL)

# Ollama's installer needs zstd, which Colab's base image doesn't ship.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

if shutil.which('ollama'):
    subprocess.Popen(['ollama', 'serve'])
    time.sleep(5)
    !ollama pull {SLM_MODEL}
    os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
    os.environ['LMTRADE_SLM_MODEL'] = SLM_MODEL
    os.environ['LMTRADE_MODEL_STACK'] = 'heuristic,slm'
    print('\nOllama serving', SLM_MODEL)
else:
    os.environ['LMTRADE_MODEL_STACK'] = 'heuristic'
    print('\nOllama not available — continuing with financial models + learned strategies only.')

## 3. (Optional) Persist state to Google Drive

The learning population, trade history and equity/alpha curves live in `data/`. Point it at Drive so **training survives disconnects** — essential if you want the 3–6-month paper-training arc rather than one-off demos.

In [ ]:
PERSIST_TO_DRIVE = True  # set False to keep state ephemeral

if PERSIST_TO_DRIVE:
    from google.colab import drive
    import os, pathlib
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/lmtrade_data'
    pathlib.Path(data_dir).mkdir(parents=True, exist_ok=True)
    if os.path.islink('data'):
        os.unlink('data')
    if not os.path.exists('data'):
        os.symlink(data_dir, 'data')
    print('State persists to', data_dir)
else:
    print('State is ephemeral (will reset on disconnect).')

## 4. Configure

Paper mode, **100 EUR budget** (flat ~1 EUR fees are a 10% drag at 10 EUR scale — 100 keeps P&L honest but readable), options + learning on. Add API keys to enable the cloud LLM vote (daily Claude review) and hourly Perplexity news.

In [ ]:
import os

os.environ['LMTRADE_MODE'] = 'paper'
os.environ['LMTRADE_BUDGET'] = '100'
os.environ['LMTRADE_UNIVERSE'] = 'AAPL,MSFT,SPY'
os.environ['LMTRADE_GPU_USD_PER_HOUR'] = '0'   # Colab free tier; Pro ~0.15

# Optional keys — uncomment to enable:
# os.environ['ANTHROPIC_API_KEY'] = '...'      # daily Claude strategy review
# os.environ['PERPLEXITY_API_KEY'] = '...'     # hourly market news

print('Configured. Model stack:', os.environ.get('LMTRADE_MODEL_STACK', 'heuristic'))

## 5. Walk-forward backtest — pre-train the strategies (dry run)

Before trading a single paper cent, pre-train the genome population on historical data. The data is split into rolling **[train | test]** folds: every genome trades the train window (its simulated P&L feeds the evolutionary optimizer), then the fittest genome is scored on the **unseen** test window. The out-of-sample column is the honest number.

The trained population **persists**, so the live engine below starts with learned fitness instead of a cold start. With yfinance installed this uses real daily history; offline it falls back to synthetic data.

In [ ]:
# ~2 years of daily bars, 150-bar train / 50-bar test rolling folds
!lmtrade backtest --bars 500 --train 150 --test 50

## 6. Demo trading run

A fast high-cadence demo: ~40 cycles at 2s intervals, starting from the pre-trained population. Watch it open synthetic option positions (calls on buy signals, puts on sell signals), manage exits, and attribute P&L to strategy genomes.

In [ ]:
!lmtrade run --cycles 40 --interval 2
print('\n--- status ---')
!lmtrade status

## 7. Dashboard — inline visualizations

Equity curve **vs the retail benchmark** (dotted) with live **alpha**, portfolio & self-sustaining economics, compute spend, positions & options, recent trades.

In [ ]:
%matplotlib inline
import lmtrade.viz as viz

viz.show()

In [ ]:
display(viz.positions_df())
display(viz.trades_df(20))
display(viz.activity_df(20))

## 8. Long-running paper training (background)

Starts the continuous learner. Genomes are evaluated on realized P&L and evolve every N closed trades; with Drive persistence (cell 3) the population keeps improving across sessions. Re-run `viz.show()` or `viz.live()` any time to check in.

In [ ]:
import subprocess

engine = subprocess.Popen(['lmtrade', 'run', '--interval', '15'])
print('Training engine running (pid', engine.pid, '), 15s cadence.')
# viz.live(interval=10, iterations=30)   # optional: watch it live

## 9. Deploy a production instance to Vast.ai

Provisions a GPU box (T4-class by default — the economics floor scales with the hourly rate you pick), installs Ollama + LMTrade, and starts the engine + web dashboard. Needs `VAST_API_KEY`.

**Reminder:** the bot halts new entries when its net worth can no longer fund `min_runway_hours` of GPU time — that guardrail is what "self-sustaining" means operationally. It reports itself self-sustaining once cumulative P&L covers all compute spend.

In [ ]:
import os
os.environ['VAST_API_KEY'] = ''   # <- fill in to deploy

if os.environ['VAST_API_KEY']:
    !pip install -q vastai
    !vastai set api-key {os.environ['VAST_API_KEY']}
    # Cheapest reliable T4 (swap gpu_name for A100_SXM4 etc.)
    !vastai search offers 'reliability>0.98 verified=true rentable=true gpu_name=Tesla_T4' --order dph_total | head -6
    print('\nPick an offer ID above, then run the deploy script:')
    print('  GPU=Tesla_T4 bash scripts/deploy_vast.sh')
    print('The onstart script installs Ollama + LMTrade and launches engine + dashboard.')
    print('Copy your .env (API keys, LMTRADE_GPU_USD_PER_HOUR=<offer rate>) to the box out-of-band.')
else:
    print('Set VAST_API_KEY above to browse offers and deploy.')

## Stop the engine

In [ ]:
proc = globals().get('engine')
if proc is not None:
    proc.terminate()
    print('stopped engine')